# 06 - Resistance patterns by patient and mutation subtype

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-HIVDB-Drug-Resistance-Maps/blob/2a10240666703507157e0faf400893343dc1c72e/content/06_resistance_stratification.ipynb)



Required approved source tables are absent or user-supplied; source workflow/conditional template must be completed first.

**Provenance:** `content/06_resistance_stratification.ipynb` at `2a10240666703507157e0faf400893343dc1c72e`; original SHA-256 `a715f8df167dd348df690312c88991e594bfdc247a6d2af27996d0739a8ead19`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 06 - Resistance patterns by patient and mutation subtype

This notebook provides the drill-down requested for ART-naive versus ART-experienced people, transmitted versus acquired resistance, mutation subtype, and drug class. The comparisons are descriptive and require a row-level analytic export with explicit denominators.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('data')
candidates = [DATA / 'hivdb_patient_resistance.csv', DATA / 'current' / 'hivdb_patient_resistance_current.csv']
path = next((p for p in candidates if p.exists()), None)
required = {'patient_id', 'iso3'}
if path is None:
    print('No patient-level resistance file found.')
    print('Expected columns include patient_id, iso3, resistant (0/1), and preferably subtype, art_status, acquisition, mutation, and drug_class.')
    resistance = pd.DataFrame()
else:
    resistance = pd.read_csv(path)
    missing = required - set(resistance.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    if 'resistant' not in resistance.columns and 'resistance_pct' not in resistance.columns:
        raise ValueError('Provide a binary resistant column or compatibility resistance_pct column.')
    if 'resistant' not in resistance.columns:
        resistance['resistant'] = pd.to_numeric(resistance['resistance_pct'], errors='coerce').div(100)
    resistance['resistant'] = pd.to_numeric(resistance['resistant'], errors='coerce')
    if not resistance['resistant'].dropna().isin([0, 1]).all():
        raise ValueError('resistant must contain only 0/1 values.')
    print(f'Loaded {len(resistance):,} rows')

## Cross-tabulate the requested strata

Use a binary person-level outcome. If the export has multiple mutation rows per person, deduplicate within the requested outcome before grouping; otherwise people with more mutations are incorrectly weighted more heavily.

In [ ]:
if not resistance.empty:
    dimensions = [c for c in ['iso3', 'subtype', 'art_status', 'acquisition', 'mutation', 'drug_class'] if c in resistance.columns]
    analysis = resistance.sort_values('resistant').drop_duplicates(['patient_id'] + dimensions, keep='last')
    if dimensions:
        grouped = (analysis.groupby(dimensions, dropna=False)
                   .agg(resistance_pct=('resistant', lambda s: 100 * s.mean()),
                        n_people=('patient_id', 'nunique'))
                   .reset_index())
        display(grouped.sort_values('n_people', ascending=False).head(50))
    else:
        print('No stratification columns were found.')
else:
    print('Stratified estimates are unavailable from the packaged 2015 study aggregates.')

## Quality checks before interpretation

Check missingness, cell sizes, and whether the same people contribute to multiple strata. Do not compare ART-naive transmitted resistance with acquired resistance in treatment-experienced populations as though they were interchangeable outcomes.

In [ ]:
if not resistance.empty:
    print('Missingness (%)')
    display((resistance.isna().mean().mul(100).sort_values(ascending=False).head(20)).to_frame('missing_pct'))
    if 'patient_id' in resistance.columns:
        print('Distinct people:', resistance['patient_id'].nunique())
    if 'art_status' in resistance.columns:
        display(resistance['art_status'].value_counts(dropna=False).to_frame('rows'))